# Relaxation, Stress, Restart, and Reference Checks

Geometry workflows connect SCF forces to structural updates.  The current scope
is still intentionally bounded:

- ion relaxation is the main path,
- orthorhombic stress is finite-difference diagnostic support,
- dense restart persistence is for small systems,
- reference comparisons are static fixtures, not runtime QE/CP2K calls.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root()


## From SCF force to structure update

For fixed-cell ion relaxation, the optimizer treats the SCF total energy as a
function of ion positions:

$$
E(R_1,R_2,\ldots,R_M).
$$

The force on ion \(I\) is the negative gradient:

$$
F_I = -\frac{\partial E}{\partial R_I}.
$$

A descent method chooses a displacement \(\Delta R\) that should lower the
energy.  A line search then accepts a step only when the trial SCF result is
finite and energetically acceptable.


## Ion-position relaxation

The optimizer reuses the previous SCF density/orbitals as continuation input.
Each accepted step records energy, force norms, step length, SCF status, and
timing summary.


### L-BFGS intuition

Steepest descent uses the force direction directly:

$$
\Delta R \propto F.
$$

L-BFGS estimates an inverse Hessian from recent changes in position and
gradient:

$$
s_k = R_{k+1}-R_k,
\qquad
y_k = \nabla E_{k+1}-\nabla E_k.
$$

That history approximates curvature without storing a full Hessian matrix.  The
implementation falls back to conservative steepest descent when the curvature
update is not trustworthy.


In [ ]:
from tempfile import TemporaryDirectory

from mlx_atomistic.dft import (
    GeometryOptimizationConfig,
    ReferenceDFTCase,
    SCFConfig,
    compare_reference_case,
    finite_difference_stress,
    geometry_demo_system,
    load_dense_scf_restart,
    optimize_geometry,
    run_scf,
    save_dense_scf_restart,
)

system = geometry_demo_system("gaussian-dimer", grid_shape=(4, 4, 4))
config = GeometryOptimizationConfig(
    max_steps=3,
    force_tolerance=1e-4,
    initial_step_size=0.06,
    scf_config=SCFConfig(max_iterations=8, solver="dense", seed=9, convergence_mode="either"),
)
relaxed = optimize_geometry(system, config=config)
print(relaxed.to_dict() | {"steps": f"{len(relaxed.steps)} accepted steps"})


In [ ]:
history = relaxed.to_dict()["history"]
energies = [row["energy"] for row in history]
max_forces = [row["max_force"] for row in history]
steps = [row["index"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(steps, energies, marker="o")
axes[0].set_title("accepted relaxation energies")
axes[0].set_xlabel("geometry step")
axes[0].set_ylabel("energy / Ha")
axes[1].semilogy(steps, max_forces, marker="s")
axes[1].set_title("maximum force")
axes[1].set_xlabel("geometry step")
axes[1].set_ylabel("force / Ha bohr$^{-1}$")
fig.tight_layout()


## Reading the relaxation trace

For a clean relaxation, energy should trend downward and the maximum force
should fall toward the configured tolerance:

$$
\max_I |F_I| \le F_\mathrm{tol}.
$$

Small non-monotonic behavior can occur if the line search permits a marginal
step or if SCF noise dominates the force scale.  Persistent force growth means
the optimizer, timestep-like step size, or SCF convergence settings need
attention.


## Diagonal stress by finite difference

The stress diagnostic samples \(E(L_x\pm\delta)\), \(E(L_y\pm\delta)\), and
\(E(L_z\pm\delta)\).  It is slow but useful for validating future analytic
stress work.


### Orthorhombic stress convention

For an orthorhombic cell with volume

$$
\Omega = L_xL_yL_z,
$$

the diagonal stress estimate used here is

$$
\sigma_{\alpha\alpha}
=-\frac{L_\alpha}{\Omega}
\frac{\partial E}{\partial L_\alpha}.
$$

The derivative is evaluated by central finite difference:

$$
\frac{\partial E}{\partial L_\alpha}
\approx
\frac{
E(L_\alpha+\delta)-E(L_\alpha-\delta)
}{2\delta}.
$$

This validates the workflow and sign conventions before we introduce an
analytic stress tensor.


In [ ]:
stress = finite_difference_stress(
    system,
    config=SCFConfig(max_iterations=2, solver="dense", seed=5, convergence_mode="either"),
    displacement=1e-2,
)
print(stress.to_dict())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["σ_xx", "σ_yy", "σ_zz"], np.asarray(stress.stress))
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title("finite-difference diagonal stress")
ax.set_ylabel("stress / Ha bohr$^{-3}$")
fig.tight_layout()


## Dense restart and static reference comparison

Restart files preserve the dense arrays needed for small-system continuation.
Reference comparisons use JSON fixtures so tests do not require QE or CP2K at
runtime.


### What a restart must preserve

A useful small-system DFT restart must preserve the objects needed to continue
SCF without changing the physical problem:

$$
\{\rho(r), \psi_i(r), f_i, R_I, \Omega, \text{k-points}, \text{spin mode}\}.
$$

The reference comparison is intentionally loose here.  It is plumbing for later
QE/CP2K fixture validation, not a claim that the current toy local/nonlocal
paths reproduce production DFT energies.


In [ ]:
scf = run_scf(system, config=SCFConfig(max_iterations=3, solver="dense", seed=12))
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "dense-restart.npz"
    save_dense_scf_restart(
        path,
        scf,
        positions=np.asarray(system.centers),
        cell_lengths=np.asarray(system.cell.lengths),
        metadata={"notebook": "relaxation-reference"},
    )
    restart = load_dense_scf_restart(path)

case = ReferenceDFTCase(
    name="toy-one-center-4x4x4",
    source="mlx-atomistic-static-reference",
    expected_energy=-0.6709365248680115,
    energy_tolerance=1.0,
)
comparison = compare_reference_case(case, observed_energy=scf.total_energy)

print("restart:", restart.to_dict())
print("reference comparison:", comparison.to_dict())
